# Розділ 7: Створення чат-додатків
## Швидкий старт з OpenAI API

Ця записна книжка адаптована з [репозиторію прикладів Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), який містить записні книжки для доступу до сервісів [Azure OpenAI](notebook-azure-openai.ipynb).

Python OpenAI API також працює з моделями Azure OpenAI, з кількома модифікаціями. Дізнайтеся більше про відмінності тут: [Як перемикатися між кінцевими точками OpenAI та Azure OpenAI за допомогою Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Огляд  
"Великі мовні моделі — це функції, які відображають текст у текст. Маючи вхідний рядок тексту, велика мовна модель намагається передбачити, який текст з’явиться далі"(1). Цей блокнот "швидкого старту" ознайомить користувачів із високорівневими поняттями великих мовних моделей, основними пакетами для початку роботи з AML, м’яким вступом до створення запитів та кількома короткими прикладами різних варіантів використання. 


## Table of Contents  

[Overview](#overview)  
[How to use OpenAI Service](#how-to-use-openai-service)  
[1. Creating your OpenAI Service](#1.-creating-your-openai-service)  
[2. Installation](#2.-installation)    
[3. Credentials](#3.-credentials)  

[Use Cases](#use-cases)    
[1. Summarize Text](#1.-summarize-text)  
[2. Classify Text](#2.-classify-text)  
[3. Generate New Product Names](#3.-generate-new-product-names)  
[4. Fine Tune a Classifier](#4.fine-tune-a-classifier)  

[References](#references)


### Створіть свій перший запит  
Це коротке вправлення надасть базове введення для надсилання запитів до моделі OpenAI для простої задачі "підсумовування".


**Кроки**:  
1. Встановіть бібліотеку OpenAI у своєму середовищі Python  
2. Завантажте стандартні допоміжні бібліотеки та встановіть свої типовi облікові дані безпеки OpenAI для створеного вами сервісу OpenAI  
3. Оберіть модель для вашої задачі  
4. Створіть простий запит для моделі  
5. Надішліть свій запит до API моделі!


### 1. Встановлення OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Імпорт допоміжних бібліотек та створення облікових даних


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Вибір відповідної моделі  
Моделі, такі як GPT-4o та GPT-4o mini, можуть розуміти та генерувати природну мову, і доступні на платформі OpenAI з різними рівнями потужності та швидкості, що підходять для різних завдань.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Проєктування запитів  

"Магія великих мовних моделей полягає в тому, що, навчаючись мінімізувати цю помилку прогнозування на величезних обсягах тексту, моделі врешті-решт вивчають концепції, корисні для цих прогнозів. Наприклад, вони вивчають такі концепції"(1):

* як правопис
* як працює граматика
* як перефразовувати
* як відповідати на запитання
* як вести розмову
* як писати багатьма мовами
* як кодувати
* тощо.

#### Як керувати великою мовною моделлю  
"З усіх вхідних даних для великої мовної моделі найбільший вплив має текстовий запит(1).

Великі мовні моделі можна спонукати до генерації виводу кількома способами:

Інструкція: Скажіть моделі, чого ви хочете
Завершення: Спонукайте модель доповнити початок того, що ви хочете
Демонстрація: Покажіть моделі, чого ви хочете, за допомогою:
Кількох прикладів у запиті
Багатьох сотень або тисяч прикладів у навчальному наборі для тонкого налаштування"



#### Існує три основні рекомендації щодо створення запитів:

**Показати і сказати**. Чітко поясніть, чого ви хочете, за допомогою інструкцій, прикладів або їх поєднання. Якщо ви хочете, щоб модель впорядкувала список елементів в алфавітному порядку або класифікувала абзац за настроєм, покажіть, що саме ви бажаєте.

**Надавайте якісні дані**. Якщо ви намагаєтеся створити класифікатор або змусити модель слідувати певному шаблону, переконайтеся, що прикладів достатньо. Обов’язково перевірте свої приклади — модель зазвичай достатньо розумна, щоб помічати базові орфографічні помилки і давати вам відповідь, але вона також може припустити, що це зроблено навмисно, і це може вплинути на відповідь.

**Перевірте свої налаштування.** Параметри temperature та top_p контролюють, наскільки детермінованою є відповідь моделі. Якщо ви очікуєте відповідь, де є лише одна правильна відповідь, то краще встановити ці параметри нижче. Якщо ви хочете отримати більш різноманітні відповіді, то можна встановити їх вищими. Найпоширеніша помилка, яку люди роблять з цими налаштуваннями, полягає в тому, що вони вважають їх контролем «розумності» чи «креативності».


Source: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Надіслати!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Повторіть той самий виклик, як порівнюються результати? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Підсумовування тексту  
#### Виклик  
Підсумуйте текст, додавши «tl;dr:» у кінці текстового фрагмента. Зверніть увагу, як модель розуміє, як виконувати низку завдань без додаткових інструкцій. Ви можете експериментувати з більш описовими підказками, ніж tl;dr, щоб змінити поведінку моделі та налаштувати отримане підсумовування(3).  

Останні дослідження продемонстрували значні досягнення на багатьох завданнях і тестах з обробки природної мови завдяки попередньому навчанню на великому корпусі тексту, а потім донавчанню на конкретному завданні. Хоча зазвичай архітектура є незалежною від завдання, цей метод все ще вимагає спеціалізованих наборів даних для донавчання з тисячами або десятками тисяч прикладів. На відміну від цього, люди зазвичай можуть виконувати нове мовне завдання, маючи лише кілька прикладів або прості інструкції — те, з чим сучасні системи NLP досі суттєво борються. Тут ми показуємо, що збільшення масштабів мовних моделей значно покращує виконання незалежних від завдання, з невеликою кількістю прикладів, іноді навіть досягаючи конкурентоспроможності з попередніми передовими методами донавчання.  



Підсумок  


# Вправи для кількох випадків використання  
1. Підсумувати текст  
2. Класифікувати текст  
3. Генерувати нові назви продуктів


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Класифікація тексту  
#### Завдання  
Класифікуйте елементи за категоріями, що надаються під час виконання. У наведеному нижче прикладі ми надаємо як категорії, так і текст для класифікації у підказці (*playground_reference). 

Запит клієнта: Вітаю, одна з клавіш на клавіатурі мого ноутбука недавно зламалася, і мені потрібна заміна:

Класифікована категорія:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Генерувати нові назви продуктів
#### Виклик
Створіть назви продуктів із прикладів слів. Тут ми включаємо в підказку інформацію про продукт, для якого ми збираємося генерувати назви. Ми також надаємо схожий приклад, щоб показати шаблон, який ми хочемо отримати. Ми також встановили високе значення температури для збільшення випадковості та більш інноваційних відповідей.

Опис продукту: Домашній міксер для молочних коктейлів
Початкові слова: швидкий, здоровий, компактний.
Назви продуктів: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис продукту: Пара взуття, яке підходить для будь-якого розміру ноги.
Початкові слова: адаптивний, підходить, омні-підгонка.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Посилання  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Найкращі практики для тонкого налаштування моделей GPT для класифікації текстів](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Для додаткової допомоги  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Учасники
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
